# 2. FranceCrops cabbage and satellite imagery

Load verified FranceCrops cabbage training footprints, select **3,000 candidate pixels**, and retrieve 2022 Sentinel-2 imagery and actual-date summaries. Then use [Four-month windows](Crop_Four_Month_Windows.ipynb) to retain up to **1,000 usable cabbage pixels**.

Run cells in order. Network access is needed for source tables and Earth Engine retrieval. Reruns reuse verified files. This notebook never downloads the large interpolated FranceCrops `x` arrays: geometry-based retrieval keeps the imagery rules consistent with Sen4AgriNet.

[Dataset card](https://huggingface.co/datasets/saget-antoine/francecrops) · [Geometry schema](https://huggingface.co/datasets/saget-antoine/francecrops/blob/main/data/geolocation/README.md)

Source labels are administrative declarations, not independently verified crop-cycle dates. Only FranceCrops `train` parcels are used. The repository revision is pinned for reproducibility.


## 1. Set local paths and pilot settings

Install the repository requirements in your notebook kernel first.

Output: `Source tables / Pilot fields / Satellite imagery: <path>` — separate FranceCrops folders.

Output: `Candidate target: 3000` — pixels before quality filtering; `Usable target: 1000` — final cap in the next notebook.


In [ ]:
from pathlib import Path
import json
import os
import francecrops_workflow
import pilot_sampling
from crop_workflow import save_json

settings_path = Path("settings.local.json")
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).resolve()
france_root = report_folder.parent / "francecrops"
revision = francecrops_workflow.DEFAULT_REVISION
source_table_folder = france_root / "source_metadata" / revision
field_folder = france_root / "fields" / "2022" / "pilot_3000"
l2a_reports = france_root / "l2a" / "2022" / "pilot_3000"
dataset_folder = (
    Path(os.environ.get("S4A_ROOT", local_settings.get("dataset_root", "./S4A")))
    .expanduser()
    .resolve()
)
l2a_folder = (
    Path(
        local_settings.get(
            "francecrops_imagery_root", str(dataset_folder / "sentinel2_francecrops")
        )
    )
    .expanduser()
    .resolve()
    / "pilot_3000"
)
timeline_folder = france_root / "analysis" / "2022" / "pilot_3000" / "timelines"
earth_engine_project = os.environ.get(
    "EE_PROJECT", local_settings.get("earth_engine_project", "YOUR_EARTH_ENGINE_PROJECT")
)
candidate_pixels_per_crop = 3000
usable_pixels_per_crop = 1000
sampling_seed = 42
cloud_probability_threshold = 40
minimum_valid_fraction = 0.8
authenticate_now = False
refresh_inventory = False
print("Source tables:", source_table_folder)
print("Pilot fields:", field_folder)
print("Satellite imagery:", l2a_folder)
print("Candidate target:", candidate_pixels_per_crop)
print("Usable target:", usable_pixels_per_crop)


## 2. Download labels and parcel footprints

Only small label tables and the approximately 55 MiB geometry table are retrieved, rather than the multi-gigabyte satellite array collection.

Output: `Reference tables: <paths>` — cached source files from the pinned revision.


In [ ]:
table_paths = francecrops_workflow.download_reference_tables(source_table_folder, revision)
print("Reference tables:", {name: str(path) for name, path in table_paths.items()})


## 3. Verify cabbage labels and select eligible parcels

Join `CHU` cabbage labels from the training split to WGS84 footprints. Invalid footprints are excluded and audited rather than silently repaired.

Output: `Valid cabbage training parcels: <number>` — parcel count, not pixel count.

Output: `Geometry audit: <path>` — original count and excluded footprints with reasons.


In [ ]:
geometry_audit_path = field_folder / "geometry_audit.json"
cabbage_features = francecrops_workflow.load_cabbage_parcels(
    table_paths, revision, geometry_audit_path
)
save_json(
    field_folder / "cabbage_training_parcels.geojson",
    {"type": "FeatureCollection", "features": cabbage_features},
)
print("Valid cabbage training parcels:", len(cabbage_features))
print("Geometry audit:", geometry_audit_path)


## 4. Select 3,000 unique candidate pixels

Use a 10 m inward parcel buffer to reduce boundary mixing. Prefer 60 spatially spread groups and add groups when small parcels cannot supply the budget, with at most 100 candidate pixels per group. Too little capacity raises an error; pixels are never duplicated to meet the target.

Output: `Interior candidate capacity: <number>` — capped candidate pool before field selection.

Output: `Selected cabbage pixels: 3000` — candidate count before cloud and window filtering.

Output: `Sampling summary: <summary>` — selected groups and candidate counts.


In [ ]:
cabbage_catalog = francecrops_workflow.build_cabbage_catalog(cabbage_features, sampling_seed)
selected_pixels = pilot_sampling.sample_candidate_pixels(
    cabbage_catalog,
    ["cabbage"],
    candidate_pixels_per_crop,
    preferred_groups=60,
    maximum_pixels_per_group=100,
    seed=sampling_seed,
)
fragments, sampling_summary = pilot_sampling.save_candidate_pilot(
    selected_pixels, field_folder, sampling_seed
)
from season_workflow import file_sha256

save_json(
    field_folder / "source_provenance.json",
    {
        "dataset": francecrops_workflow.DATASET_NAME,
        "revision": revision,
        "year": 2022,
        "class_code": "CHU",
        "source_split": "train",
        "label_status": "reported",
        "interpolated_arrays_used": False,
        "license": "CC-BY-4.0",
        "reference_tables": {
            name: {"path": str(path), "sha256": file_sha256(path)}
            for name, path in table_paths.items()
        },
    },
)
print("Interior candidate capacity:", len(cabbage_catalog))
print("Selected cabbage pixels:", len(selected_pixels))
print("Sampling summary:", sampling_summary)


## 5. Connect to Earth Engine

Set `authenticate_now = True` in Step 1 only when a new login is needed. Credentials are managed outside the notebook.

Output: `Earth Engine project: <value>` — registered project used for requests.


In [ ]:
import ee
from satellite_workflow import SatelliteRetriever

if authenticate_now:
    ee.Authenticate()
ee.Initialize(project=earth_engine_project)
retriever = SatelliteRetriever(ee, 2022, l2a_folder, l2a_reports, cloud_probability_threshold)
print("Earth Engine project:", earth_engine_project)


## 6. Inventory acquisitions and plan composites

Retrieve full-year 2022 imagery so the original interpolated February–November array coverage does not constrain detection. Annual crop labels still do not prove which crop occupied every date.

Output: `Inventoried regions: <number>` — successful parcel inventories.

Output: `Planned composite files: <number>` — 24 half-month files per successful region.

Output: `Inventory errors: <number>` — failed regions retained in `composite_plan.json` for retry.


In [ ]:
acquisition_plans, inventory_errors = retriever.build_plans(fragments, refresh=refresh_inventory)
print("Inventoried regions:", len(acquisition_plans))
print("Planned composite files:", len(acquisition_plans) * 24)
print("Inventory errors:", len(inventory_errors))


## 7. Download cloud-screened satellite composites

SCL, exact cloud-probability matches and valid scene edges are applied before daily mosaics and half-month medians. These are the same rules used for Sen4AgriNet.

Output: `<region> <interval> <status>` — per-file progress.

Output: `Downloaded / Reused / Failed composites: <number>` — new files, checksum-verified existing files, and requests needing retry. Failed and empty intervals remain explicit.


In [ ]:
download_summary = retriever.download_composites(acquisition_plans)
print("Downloaded composites:", download_summary["downloaded"])
print("Reused composites:", download_summary["reused"])
print("Failed composites:", download_summary["failed"])


## 8. Retrieve actual-date vegetation summaries

Small tables of NDVI, NDRE and NDMI support observed timing detection; this does not download additional rasters. Missing daily support cannot be replaced with interpolated benchmark arrays.

Output: `<region> <number> observed daily summaries` — actual satellite dates summarized per selected region.

Output: `Observed summary rows: <number>` — combined region-date count. Failures are retained in `observed_errors.json`.


In [ ]:
import observed_timelines

observed_rows = observed_timelines.retrieve_observed_timelines(
    acquisition_plans, retriever.mask_acquisition, ee, timeline_folder, minimum_valid_fraction
)
print("Observed summary rows:", len(observed_rows))


## 9. Save the source handoff

Output: `FranceCrops source handoff: <path>` — paths and observation year loaded by the four-month notebook. Quality checks there determine actual usable counts.

Continue with [Four-month windows](Crop_Four_Month_Windows.ipynb). The combined prototype target is **1,000 usable pixels each for carrot, cabbage and potato**. A shortfall remains a shortfall; no samples are fabricated.


In [ ]:
from pilot_sources import save_source_manifest

source_handoff_path = report_folder.parent / "source_handoffs" / "francecrops_3000.json"
source_manifest = save_source_manifest(
    source_handoff_path,
    "francecrops",
    2022,
    field_folder,
    l2a_folder,
    l2a_reports / "composite_progress",
    timeline_folder,
    revision,
)
print("FranceCrops source handoff:", source_handoff_path)
